# Training a circuit: VQE
**Quantum Computing · Week 7 (Variational quantum circuits) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w7d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**VQE**, the variational quantum eigensolver: a circuit with knobs prepares a trial state, the quantum
computer measures its energy, and a classical optimizer turns the knobs downhill. Today's target is
a real molecule, H₂, written as a two-qubit Hamiltonian.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

torch.set_num_threads(1)
# H2 at a bond length of 0.735 angstrom, mapped to two qubits (the example in Qiskit's VQE documentation);
# energies in hartree, without the repulsion between the two nuclei
H = SparsePauliOp.from_list([("II", -1.052373245772859), ("IZ", 0.39793742484318045), ("ZI", -0.39793742484318045),
                             ("ZZ", -0.01128010425623538), ("XX", 0.18093119978423156)])
exact = np.linalg.eigvalsh(H.to_matrix()).min()
print(f"exact ground-state energy: {exact:.6f} hartree")

def make_ansatz(with_cnot=True):
    knobs = ParameterVector("θ", 4)
    qc = QuantumCircuit(2)
    qc.ry(knobs[0], 0)
    qc.ry(knobs[1], 1)
    if with_cnot:
        qc.cx(0, 1)
    qc.ry(knobs[2], 0)
    qc.ry(knobs[3], 1)
    return qc

ansatz = make_ansatz()
estimator = StatevectorEstimator()

def energies(circuit, parameter_sets):
    """One Estimator call for many knob settings at once."""
    return estimator.run([(circuit, H, np.atleast_2d(parameter_sets))]).result()[0].data.evs

print("energy with all knobs at 0:", round(float(energies(ansatz, [0, 0, 0, 0])[0]), 4))

## The circuit inside PyTorch
A custom `torch.autograd.Function`: forward measures the energy, backward returns the parameter-shift
gradient (all $2 \times 4$ shifted circuits in one Estimator call). After that, it's the usual loop with Adam.

In [ ]:
class Energy(torch.autograd.Function):
    @staticmethod
    def forward(ctx, knobs, circuit):
        ctx.save_for_backward(knobs)
        ctx.circuit = circuit
        return torch.tensor(float(energies(circuit, knobs.detach().numpy())[0]), dtype=knobs.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (knobs,) = ctx.saved_tensors
        k = knobs.detach().numpy()
        shifts = np.pi / 2 * np.eye(len(k))
        e = energies(ctx.circuit, np.vstack([k + shifts, k - shifts]))      # 2P circuits
        grad = (e[: len(k)] - e[len(k):]) / 2                                 # the parameter-shift rule
        return grad_output * torch.tensor(grad, dtype=knobs.dtype), None

def vqe(circuit, steps=120, lr=0.1, seed=0):
    torch.manual_seed(seed)
    knobs = (0.1 * torch.randn(circuit.num_parameters, dtype=torch.float64)).requires_grad_()
    optimizer = torch.optim.Adam([knobs], lr=lr)
    history = []
    for _ in range(steps):
        optimizer.zero_grad()
        loss = Energy.apply(knobs, circuit)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return knobs.detach().numpy(), history

knobs, history = vqe(ansatz)
print(f"VQE energy {history[-1]:.6f}, exact {exact:.6f}, error {abs(history[-1] - exact) * 1000:.3f} millihartree")

Chemists call an error below 1.6 millihartree **chemical accuracy**: good enough to predict how
reactions behave.

In [ ]:
plt.plot(history, color="#7b5cfa", lw=2.5, label="VQE")
plt.axhline(exact, color="#17a98c", ls="--", label="exact ground state")
plt.axhspan(exact, exact + 0.0016, color="#17a98c", alpha=0.15, label="chemical accuracy")
plt.xlabel("optimizer step")
plt.ylabel("energy (hartree)")
plt.ylim(exact - 0.05, exact + 0.6)
plt.legend()
plt.show()

## Your turn: what does the CNOT buy?
Train the same ansatz **without** the CNOT. It can only make product states, one independent state
per qubit. How far above the exact energy does it get stuck?

In [ ]:
no_cnot = make_ansatz(with_cnot=False)
# TODO: run vqe(no_cnot) and print its final energy and its error in millihartree

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
no_cnot = make_ansatz(with_cnot=False)
_, history_product = vqe(no_cnot)
print(f"without the CNOT: {history_product[-1]:.6f}, error {abs(history_product[-1] - exact) * 1000:.1f} millihartree")
print(f"with the CNOT:    {history[-1]:.6f}, error {abs(history[-1] - exact) * 1000:.3f} millihartree")
print("The best product state misses about 20 millihartree: the 'correlation energy' of H2, which only an")
print("entangled state can capture. Without entanglement, no chemical accuracy.")